# 01 — Feature Inventory (CIC-IDS2018 x UNSW-NB15)

Tujuan: mengumpulkan **daftar fitur nyata** dari kedua dataset sebagai dasar **Semantic Feature Mapping** (irisan fitur penuh vs fitur penuh).

- CIC-IDS2018: 68 fitur bersih diambil dari `cleaned_100.pkl` (`feature_names`).
- UNSW-NB15: 42 fitur dari header berkas partisi.

Output: `feature_inventory.json` (daftar fitur kedua dataset) untuk diunduh & dipakai menyusun tabel mapping.

> Jalankan di SageMaker (tempat `cleaned_100.pkl` tersedia). Sesuaikan `CIC_PKL` bila path berbeda.

In [ ]:
# --- Bootstrap: pastikan dependency ada (SageMaker mereset pip saat stop/start) ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('scikit-learn','sklearn')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pickle, os, json
import pandas as pd

# --- Path (sesuaikan bila perlu) ---
CIC_PKL = '../../CICDDoS2018/data/cleaned_100.pkl'   # dari unswnb-15/notebooks/
UNSW_TRAIN = '../data/UNSW_NB15_training-set.csv'
UNSW_TEST  = '../data/UNSW_NB15_testing-set.csv'
OUT_JSON   = '../feature_inventory.json'

print('CIC pkl exists :', os.path.exists(CIC_PKL))
print('UNSW train     :', os.path.exists(UNSW_TRAIN))
print('UNSW test      :', os.path.exists(UNSW_TEST))

In [ ]:
# --- CIC-IDS2018: 68 fitur bersih ---
# cleaned_100.pkl memuat objek sklearn (StandardScaler/LabelEncoder), jadi butuh sklearn.
# Fallback: bila load penuh gagal, pakai unpickler yang men-skip objek sklearn
# sehingga 'feature_names' (list string murni) tetap terbaca.
cic_features = None

class _SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith('sklearn') or module.startswith('numpy'):
            try:
                return super().find_class(module, name)
            except Exception:
                return type('_Stub', (), {})  # stub bila kelas tak tersedia
        return super().find_class(module, name)

if os.path.exists(CIC_PKL):
    try:
        with open(CIC_PKL, 'rb') as f:
            d = pickle.load(f)
        cic_features = list(d['feature_names'])
    except ModuleNotFoundError as e:
        print(f'[warn] load penuh gagal ({e}); mencoba fallback unpickler ...')
        with open(CIC_PKL, 'rb') as f:
            d = _SafeUnpickler(f).load()
        cic_features = list(d['feature_names'])
    print(f'CIC-IDS2018: {len(cic_features)} fitur')
    for i, c in enumerate(cic_features, 1):
        print(f'  {i:2d}. {c}')
else:
    print('cleaned_100.pkl tidak ditemukan - jalankan di SageMaker atau perbaiki path CIC_PKL')

In [ ]:
# --- UNSW-NB15: 42 fitur (buang id, attack_cat, label) ---
df = pd.read_csv(UNSW_TRAIN, nrows=5)
drop = {'id', 'attack_cat', 'label'}
unsw_features = [c for c in df.columns if c not in drop]
print(f'UNSW-NB15: {len(unsw_features)} fitur')
for i, c in enumerate(unsw_features, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
# --- Statistik ringkas UNSW (untuk validasi mapping nanti) ---
full = pd.read_csv(UNSW_TRAIN)
num = full[unsw_features].select_dtypes(include='number')
desc = num.describe().T[['min', '25%', '50%', '75%', 'max', 'mean', 'std']]
print('Statistik ringkas fitur numerik UNSW-NB15 (data latih):')
print(desc.to_string())

In [ ]:
# --- Simpan inventory ke JSON (untuk diunduh & dipakai menyusun mapping) ---
inventory = {
    'cic_ids2018': {
        'n_features': len(cic_features) if cic_features else None,
        'features': cic_features,
        'extractor': 'CICFlowMeter',
    },
    'unsw_nb15': {
        'n_features': len(unsw_features),
        'features': unsw_features,
        'extractor': 'Argus + Bro/Zeek (+12 custom algorithms)',
    },
}
with open(OUT_JSON, 'w') as f:
    json.dump(inventory, f, indent=2)
print('Saved:', OUT_JSON)
print('\nRingkas:')
print(f"  CIC-IDS2018 : {inventory['cic_ids2018']['n_features']} fitur (CICFlowMeter)")
print(f"  UNSW-NB15   : {inventory['unsw_nb15']['n_features']} fitur (Argus/Bro)")
print('\nUnduh feature_inventory.json lalu kabari isinya untuk menyusun tabel Semantic Feature Mapping.')